## Hw 1b: Experimental Analysis of a Shallow Neural Network

Import all necessary libraries

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import torch

from training_utils import run_experiment, plot_results, apply_augmentations, show_batch

### 1. Validation set
First 10,000 images of the training set for validation, the remaining 50,000 for training

In [ ]:
train_df = pd.read_csv("fashion_mnist/fashion-mnist_train.csv")

labels_np = train_df.iloc[:, 0].astype(np.int64).values.copy()
images_np = train_df.iloc[:, 1:].astype(np.float32).values.copy()

X = torch.tensor(images_np, dtype=torch.float32) / 255.0
Y = torch.tensor(labels_np, dtype=torch.long)

X_valid, Y_valid = X[:10000], Y[:10000]
X_train, Y_train = X[10000:], Y[10000:]

X_train.shape, X_valid.shape

### 2. Learning rate selection
Hidden size 50, 500 epochs, new model for every learning rate. Same seed before each model so they all start from the same weights

In [ ]:
epochs = 500
learning_rates = [1, 0.1, 0.01, 0.001, 0.0001, 0.00001]

lr_results = [run_experiment(X_train, Y_train, X_valid, Y_valid,
                             learning_rate=lr, hidden_size=50, epochs=epochs)
              for lr in learning_rates]
lr_df = pd.DataFrame(lr_results)[["learning_rate", "train_loss", "valid_loss", "valid_acc"]]
lr_df

Interpolate between the two best learning rates. We rank them by validation loss instead of accuracy, it is smoother and also shows when a model starts overfitting

In [ ]:
low = lr_df.nsmallest(2, "valid_loss")["learning_rate"].min()
new_lrs = [low * m for m in [2, 3, 5, 7]]
print("new learning rates:", new_lrs)

new_results = [run_experiment(X_train, Y_train, X_valid, Y_valid,
                              learning_rate=lr, hidden_size=50, epochs=epochs)
               for lr in new_lrs]
lr_all_df = pd.concat([lr_df, pd.DataFrame(new_results)[lr_df.columns]])
lr_all_df = lr_all_df.sort_values("learning_rate").reset_index(drop=True)

best_lr = lr_all_df.loc[lr_all_df["valid_loss"].idxmin(), "learning_rate"]
print("best learning rate:", best_lr)
lr_all_df

In [ ]:
plot_results(lr_all_df, "learning_rate", log_x=True)

### 3. Effect of hidden layer size
Learning rate fixed to the best one from above

In [ ]:
hidden_sizes = [10, 50, 100, 300, 1000, 2000]

hidden_results = [run_experiment(X_train, Y_train, X_valid, Y_valid,
                                 learning_rate=best_lr, hidden_size=h, epochs=epochs)
                  for h in hidden_sizes]
hidden_df = pd.DataFrame(hidden_results)[["hidden_size", "train_loss", "valid_loss", "valid_acc"]]

best_hidden = int(hidden_df.loc[hidden_df["valid_loss"].idxmin(), "hidden_size"])
print("best hidden size:", best_hidden)
hidden_df

In [ ]:
plot_results(hidden_df, "hidden_size", log_x=True)

### 4. Visualize transformed images
First row: original images. Rows 2 to 4: mild, moderate and aggressive

In [ ]:
imgs = X_train[:4]

# one call per image so every image gets its own random transform
rows = [imgs]
for level in ["mild", "moderate", "aggressive"]:
    rows.append(torch.cat([apply_augmentations(img, level=level) for img in imgs]))

show_batch(torch.cat(rows).reshape(-1, 28, 28))

### 5. Effect of data augmentation
Best learning rate and hidden size from above. Only the training images are augmented, so the train loss of the augmented runs is measured on augmented images

In [ ]:
aug_levels = [None, "mild", "moderate", "aggressive"]

aug_results = [run_experiment(X_train, Y_train, X_valid, Y_valid,
                              learning_rate=best_lr, hidden_size=best_hidden, epochs=epochs, level=level)
               for level in aug_levels]
aug_df = pd.DataFrame(aug_results)[["augmentation", "train_loss", "valid_loss", "valid_acc"]]
aug_df

In [ ]:
plot_results(aug_df, "augmentation", kind="bar")